# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. The dataset includes clinical and pathological variables for 77 cancer survivors diagnosed with a second primary colorectal cancer, accessible via a Croissant schema. The steps follow the [mlcroissant usage template](https://github.com/mlcommons/croissant/blob/main/python/examples/explore_toy.ipynb), referencing all data entities by their `@id` as required for reproducibility and clarity.

### Dataset Source
The Croissant schema for this dataset is available at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load Croissant metadata and medical records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print("Dataset Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Published:", dataset.metadata.datePublished)
print("License:", dataset.metadata.license)
print("Identifier:", dataset.metadata.identifier)


## 2. Data Overview

We'll enumerate the available record sets and their fields by their `@id` as described by the Croissant schema. This helps understand the structure and granularity of the dataset.

> **Note:** Croissant record sets and fields are referenced by their `@id` fields, which must be used for all data access in this notebook.

In [ ]:
# Explore available record sets in the dataset schema.
# The Croissant spec exposes .record_sets for Dataset objects.

record_sets = dataset.record_sets

print("Available record sets (by @id):")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    if rs.get('field'):
        print("   Fields:")
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"      - @id: {field['@id']}")
            else:
                print(f"      - @id: {field}")

Let's also preview the beginning of the tabular/case record set. You'll need the specific `@id` values for use in extraction below.

In [ ]:
# Let's print the first record for each record set using their @id.

for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nFirst record for record set @id: {rs_id}")
    try:
        for i, record in enumerate(dataset.records(record_set=rs_id)):
            print(record)
            if i >= 0:
                break
    except Exception as e:
        print(f"Could not extract records for {rs_id}: {e}")

## 3. Data Extraction

We extract each record set into a pandas DataFrame for analysis. Please note that all names are referenced by their full `@id` values.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

# Load record sets into DataFrames by their @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Extracting records for record set: {record_set_id}")
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f"Columns in {record_set_id}:", df.columns.tolist())

# For further demonstration, we'll pick the main clinical tabular record set -- typically the first/primary.
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"\nUsing record set: {main_record_set_id} for EDA.")
    print("Sample records:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We illustrate basic filtering, normalization, and group analysis. All field names are referenced exactly as `@id` strings from the schema. Please consult the DataFrame columns above to select suitable numeric and group fields. Here, we select `'age'` as a clinical numeric field and `'sex'` as a group field (use the actual `@id` from your dataset).

In [ ]:
# Set record and field IDs (replace below with the actual @ids from your schema if different)

# Example - suppose the @ids for age and sex are:
main_record_set_id = record_set_ids[0]

df = dataframes[main_record_set_id]

print("Fields available in selected record set:", df.columns.tolist())

# Replace these with exact @id values from your schema/columns
numeric_field_id = None
group_field_id = None

# Find a numeric field: typically 'age' or equivalent. Try to auto-detect an 'age' column.
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break

# Find a grouping/categorical field: e.g., 'sex' or 'msi_status', etc.
for col in df.columns:
    if 'sex' in col.lower() or 'gender' in col.lower():
        group_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field (e.g., 'age') found for analysis. Please check schema.")
else:
    print(f"\nNumeric field chosen (by @id): {numeric_field_id}")

if group_field_id is None:
    print("No group field (e.g., 'sex') found for grouping. Group analysis will be skipped.")
else:
    print(f"Group field chosen (by @id): {group_field_id}")

# Perform filtering, normalization, and group stats (if possible)
if numeric_field_id is not None:
    # Remove impossible ages (for illustration; threshold below 15 y old)
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = 15
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered cases with {numeric_field_id} > {threshold}")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered cases:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If a group_field is found, show group statistics
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} by {group_field_id}:")
            display(grouped_df)
    except Exception as e:
        print(f"Error during EDA: {e}")

## 5. Visualization

Now we'll visualize some relationships in the data, such as a histogram of age distribution and, if available, how age varies by sex or MSI status. Make sure relevant libraries are imported.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram of numeric variable (e.g., age)
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by group field (e.g., sex)
if group_field_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(6,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

This notebook demonstrates loading, overview, and basic exploratory analysis of the FAIR² clinical CRC survivors dataset using `mlcroissant`, referencing all Croissant entities by their `@id`. 

- We loaded the Croissant package and observed record sets, fields, and records using schema-origin IDs.
- Simple exploration, filtering, normalization, and group analysis on numeric/clinical fields were performed.
- Visualizations helped reveal variable distributions and groupwise differences.

This approach can be extended to deeper modeling and validation tasks. For other record sets, repeat the above with their respective `@id`s and fields.